In [ ]:
import random
import hashlib
from dataclasses import dataclass

@dataclass
class User:
    user_id: str
    ltv: float
    days_since_last_purchase: int
    intent_score: float
    price_sensitivity: float
    cart_value: float
    is_new_user: bool


In [ ]:
def get_rng(user_id: str):
    seed = int(hashlib.md5(user_id.encode()).hexdigest()[:8], 16)
    return random.Random(seed)


In [ ]:
def get_mean(user: User):
    mean = 20

    if user.intent_score < 0.3:
        mean += 3
    elif user.intent_score > 0.7:
        mean -= 3

    if user.ltv > 1000:
        mean -= 2
    elif user.ltv < 200:
        mean += 2

    if user.days_since_last_purchase > 90:
        mean += 2

    if user.price_sensitivity > 0.7:
        mean += 2

    if user.cart_value > 200:
        mean -= 2

    if user.is_new_user:
        mean += 1

    return mean


In [ ]:
def assign_discount(user: User):
    rng = get_rng(user.user_id)
    mean = get_mean(user)
    std_dev = 5

    score = rng.gauss(mean, std_dev)

    if score < 15:
        return 10
    elif score < 25:
        return 20
    else:
        return 30


In [ ]:
users = [
    User(f"user_{i}", 100, 30, 0.4, 0.6, 100, False)
    for i in range(10)
]

for u in users:
    print(u.user_id, assign_discount(u))


In [ ]:
users = [
    User(
        user_id=f"user_{i}",
        ltv=random.randint(0, 2000),
        days_since_last_purchase=random.randint(0, 365),
        intent_score=random.random(),
        price_sensitivity=random.random(),
        cart_value=random.randint(10, 500),
        is_new_user=random.choice([True, False])
    )
    for i in range(1000)
]

results = {10: 0, 20: 0, 30: 0}

for u in users:
    results[assign_discount(u)] += 1

print(results)

total = sum(results.values())
for k in sorted(results):
    pct = results[k] / total
    bar = '█' * int(pct * 50)
    print(f"{k}% | {bar} ({results[k]}, {pct:.1%})")
